# LSTM Patient Deterioration Warning
## Can a neural network remember an important warning from hours ago?

> **Educational demo only.** This project uses synthetic data and is not a clinical prediction tool.

We will use **24 hours of vital signs** to predict whether a patient belongs to a deterioration pattern.

```text
24 hours × 3 vital signs → RNN / LSTM → deterioration risk
```

Vital signs: **heart rate, respiratory rate, SpO₂**.


# 1. Load the dataset

The dataset is already stored in this GitHub repository. We simply download it with Pandas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

DATA_URL = 'https://raw.githubusercontent.com/dujing82-blip/tiny-lstm-patient-deterioration/main/patient_vital_signs.csv'
data = pd.read_csv(DATA_URL)

print(data.head())
print('\nRows:', len(data))
print('Patients:', data['patient_id'].nunique())


# 2. Look at one patient's 24-hour history

A sequence model does not see just one measurement. It sees how the measurements **change over time**.


In [ ]:
patient = data[data['patient_id'] == 750]

plt.figure(figsize=(12,4))
plt.plot(patient['hour'], patient['heart_rate_bpm'], marker='o', label='Heart rate')
plt.plot(patient['hour'], patient['respiratory_rate_bpm'], marker='o', label='Respiratory rate')
plt.plot(patient['hour'], patient['spo2_percent'], marker='o', label='SpO₂')
plt.xlabel('Hour')
plt.ylabel('Measurement')
plt.title('One Patient: 24 Hours of Vital Signs')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print('Case type:', patient['case_type'].iloc[0])
print('Deterioration label:', patient['deterioration'].iloc[0])


# 3. Turn the table into RNN sequences

Each patient becomes one training example:

```text
24 time steps
     ×
3 features at every time step

X.shape = (patients, 24, 3)
```


In [ ]:
FEATURES = ['heart_rate_bpm', 'respiratory_rate_bpm', 'spo2_percent']

X = []
y = []
case_types = []

for patient_id, patient in data.groupby('patient_id'):
    X.append(patient[FEATURES].to_numpy())
    y.append(patient['deterioration'].iloc[0])
    case_types.append(patient['case_type'].iloc[0])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)
case_types = np.array(case_types)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nOne patient =', X[0].shape, '= 24 hours × 3 vital signs')


# 4. Train / test split and normalization

We keep 20% of the patients completely unseen for testing. We calculate normalization values using the training patients only.


In [ ]:
# Shuffle PATIENTS, not individual hours.
rng = np.random.default_rng(7)
indices = rng.permutation(len(X))
split = int(0.8 * len(X))
train_idx = indices[:split]
test_idx = indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
case_test = case_types[test_idx]

# Mean and standard deviation for each of the 3 vital signs.
mean = X_train.mean(axis=(0,1), keepdims=True)
std = X_train.std(axis=(0,1), keepdims=True)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print('Training patients:', len(X_train))
print('Testing patients:', len(X_test))


# 5. First try: SimpleRNN

This is the model from our previous lesson. It reads the 24 hours in order and carries a hidden state forward.


In [ ]:
rnn_model = keras.Sequential([
    layers.Input(shape=(24,3)),
    layers.SimpleRNN(16),
    layers.Dense(1, activation='sigmoid')
])

rnn_model.compile(
    optimizer='adam',
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

rnn_model.summary()

rnn_model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


# 6. Now replace RNN with LSTM

Look carefully: the model is almost identical. We change essentially **one layer**.

```text
SimpleRNN(16)  →  LSTM(16)
```

LSTM was designed to preserve useful information over longer sequences.


In [ ]:
lstm_model = keras.Sequential([
    layers.Input(shape=(24,3)),
    layers.LSTM(16),                 # <-- the important change
    layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss=keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

lstm_model.summary()

lstm_model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


# 7. Compare RNN and LSTM


In [ ]:
rnn_prob = rnn_model.predict(X_test, verbose=0).flatten()
lstm_prob = lstm_model.predict(X_test, verbose=0).flatten()

rnn_pred = (rnn_prob >= 0.5).astype(int)
lstm_pred = (lstm_prob >= 0.5).astype(int)

rnn_accuracy = np.mean(rnn_pred == y_test)
lstm_accuracy = np.mean(lstm_pred == y_test)

print(f'SimpleRNN overall accuracy: {rnn_accuracy:.3f}')
print(f'LSTM overall accuracy:      {lstm_accuracy:.3f}')


# 8. The important test: early-warning patients

Some deterioration examples contain an important abnormal event **early** in the 24-hour history, followed by partial recovery.

These cases ask the model to remember information that is farther in the past.


In [ ]:
early = (case_test == 'early_warning')

rnn_early_accuracy = np.mean(rnn_pred[early] == y_test[early])
lstm_early_accuracy = np.mean(lstm_pred[early] == y_test[early])

print('Number of early-warning test patients:', early.sum())
print(f'SimpleRNN accuracy on early-warning cases: {rnn_early_accuracy:.3f}')
print(f'LSTM accuracy on early-warning cases:      {lstm_early_accuracy:.3f}')


# 9. Visual comparison


In [ ]:
labels = ['Overall', 'Early warning']
rnn_scores = [rnn_accuracy, rnn_early_accuracy]
lstm_scores = [lstm_accuracy, lstm_early_accuracy]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8,5))
plt.bar(x - width/2, rnn_scores, width, label='SimpleRNN')
plt.bar(x + width/2, lstm_scores, width, label='LSTM')
plt.xticks(x, labels)
plt.ylim(0,1.05)
plt.ylabel('Accuracy')
plt.title('SimpleRNN vs. LSTM')
plt.legend()
plt.show()


# 10. Why LSTM?

The key question is not simply **“Which model has higher accuracy?”**

The deeper question is:

> **Can the network preserve important information when it happened many time steps ago?**

```text
Hour 4: important warning
       ↓
       ───────── many later measurements ─────────→ Hour 24
                                                    ↓
                                               prediction
```

A SimpleRNN carries a hidden state, but information can become difficult to preserve over long sequences. LSTM adds a **cell state and gates** that control what information should be remembered, updated, or forgotten.

That is why our next question is: **How do the LSTM gates work?**

> Reminder: this synthetic classroom exercise demonstrates sequence-model concepts only. It must not be interpreted as a clinical deterioration score or used for patient care.


# 11. Try it yourself: create a change in the patient's vital signs

Changing **one single hour** may not move an LSTM very much. That is actually an important lesson: this model was trained to recognize **patterns across time**, not isolated measurements.

In this playground, choose a patient and then create a change that lasts for several hours.

For example:

- Select **Hours 4–9**.
- Increase heart rate by **+25 bpm**.
- Increase respiratory rate by **+7 breaths/min**.
- Decrease SpO₂ by **−5%**.
- Click **Apply change**.

Then move the same pattern to **Hours 18–23** and compare the probability. This lets you ask two different questions:

> **How much does the duration of an abnormal pattern matter?**

> **Does the timing of the same pattern matter to the LSTM?**

The display also shows the **original probability** next to the modified probability, so students can see exactly how much their intervention changed the model output.

> **Classroom simulation only:** this is synthetic educational data and a toy neural network. The displayed probability is not a clinical risk score and must not be used for patient care.


In [ ]:
# Interactive LSTM playground: edit a TIME WINDOW, not just one point.
# Run this cell AFTER training the LSTM above.

import ipywidgets as widgets
from IPython.display import display, clear_output

patient_slider = widgets.IntSlider(
    value=0, min=0, max=min(49, len(X_test)-1), step=1,
    description='Patient:', continuous_update=False
)

start_hour = widgets.IntSlider(
    value=4, min=1, max=24, step=1,
    description='Start hour:', continuous_update=False
)

end_hour = widgets.IntSlider(
    value=9, min=1, max=24, step=1,
    description='End hour:', continuous_update=False
)

# These sliders represent CHANGES from the original patient.
hr_change = widgets.IntSlider(
    value=0, min=-30, max=50, step=1,
    description='HR change:', continuous_update=False
)

rr_change = widgets.IntSlider(
    value=0, min=-10, max=20, step=1,
    description='RR change:', continuous_update=False
)

spo2_change = widgets.IntSlider(
    value=0, min=-12, max=5, step=1,
    description='SpO₂ change:', continuous_update=False
)

apply_button = widgets.Button(description='Apply change', button_style='warning')
reset_button = widgets.Button(description='Reset patient')
early_button = widgets.Button(description='Example: early warning')
late_button = widgets.Button(description='Example: late warning')
output = widgets.Output()

original_patient = X_test[0].copy()
current_patient = original_patient.copy()

def predict_probability(patient_history):
    # X_test is already normalized in this notebook, so a raw patient history
    # must be normalized with the SAME training mean and standard deviation.
    scaled = (patient_history - mean) / std
    scaled = scaled[np.newaxis, ...]
    return float(lstm_model.predict(scaled, verbose=0)[0, 0])

def draw_patient():
    original_p = predict_probability(original_patient)
    modified_p = predict_probability(current_patient)
    difference = modified_p - original_p
    hours = np.arange(1, 25)

    with output:
        clear_output(wait=True)

        print(f'Original LSTM probability: {original_p:.1%}')
        print(f'Modified LSTM probability: {modified_p:.1%}')
        print(f'Change in probability:     {difference:+.1%}')

        if modified_p >= 0.5:
            print('\n⚠️ MODEL WARNING: DETERIORATION')
        else:
            print('\n✓ MODEL PREDICTION: STABLE')

        print('\nSynthetic classroom model only — not a clinical risk score.')

        fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
        names = ['Heart rate (bpm)', 'Resp. rate (breaths/min)', 'SpO₂ (%)']

        for i, ax in enumerate(axes):
            ax.plot(hours, original_patient[:, i], '--', alpha=0.45, label='Original')
            ax.plot(hours, current_patient[:, i], marker='o', label='Modified')
            ax.set_ylabel(names[i])
            ax.grid(alpha=0.2)
            ax.legend(loc='best')

        axes[2].set_xlabel('Hour')
        axes[2].set_xticks(np.arange(1, 25))
        plt.suptitle('Original vs. Modified 24-Hour History')
        plt.tight_layout()
        plt.show()

def load_patient(number):
    global original_patient, current_patient
    original_patient = X_test_raw[number].copy()
    current_patient = original_patient.copy()
    draw_patient()

def apply_change(button):
    global current_patient
    a = min(start_hour.value, end_hour.value) - 1
    b = max(start_hour.value, end_hour.value)  # Python slice stops after this hour

    # Start again from the original patient so every click is easy to interpret.
    current_patient = original_patient.copy()
    current_patient[a:b, 0] += hr_change.value
    current_patient[a:b, 1] += rr_change.value
    current_patient[a:b, 2] += spo2_change.value

    # Keep values within the classroom simulator's displayed range.
    current_patient[:, 0] = np.clip(current_patient[:, 0], 45, 160)
    current_patient[:, 1] = np.clip(current_patient[:, 1], 6, 45)
    current_patient[:, 2] = np.clip(current_patient[:, 2], 80, 100)

    draw_patient()

def reset_patient(button):
    global current_patient
    current_patient = original_patient.copy()
    hr_change.value = 0
    rr_change.value = 0
    spo2_change.value = 0
    draw_patient()

def early_example(button):
    start_hour.value = 4
    end_hour.value = 9
    hr_change.value = 25
    rr_change.value = 7
    spo2_change.value = -5
    apply_change(None)

def late_example(button):
    start_hour.value = 18
    end_hour.value = 23
    hr_change.value = 25
    rr_change.value = 7
    spo2_change.value = -5
    apply_change(None)

def patient_changed(change):
    load_patient(change['new'])

patient_slider.observe(patient_changed, names='value')
apply_button.on_click(apply_change)
reset_button.on_click(reset_patient)
early_button.on_click(early_example)
late_button.on_click(late_example)

display(widgets.HTML('<h3>LSTM Patient Playground</h3>'))
display(patient_slider)
display(widgets.HTML('<b>Choose a time window:</b>'))
display(widgets.HBox([start_hour, end_hour]))
display(widgets.HTML('<b>Change the vital signs during that window:</b>'))
display(hr_change, rr_change, spo2_change)
display(widgets.HBox([apply_button, reset_button]))
display(widgets.HTML('<br><b>Quick experiments:</b>'))
display(widgets.HBox([early_button, late_button]))
display(output)

# The notebook keeps raw and normalized test data separately.
load_patient(0)
